# MiniMax H3 — Colab生成ノートブック（窓際族物語 / colab-video スキル・製品生成専用）

動画スキルで制作済みのバンドル（キーフレーム＋wav＋`ch*_workflow.json`＋`h3_run.py`）を、
ColabのGPU（L4 / A100）でチャプター毎に動画化する。配管検証は2026-08に完了済みのため、このノートブックは生成だけに特化している。

使い方: **セル1だけ編集**したら、その直下の**★一括実行セル**を押す（セル1〜7を続けて流す＝1つずつ押す必要はない）。途中で失敗したら原因を直し、そのセルの`FROM_STEP`を失敗した番号にして再実行すれば続きから流せる。1つずつ確認しながら進めたいときは従来どおりセル1→8を順に実行してもよい。パイロット（セリフ有りチャプター）を先に1本生成して確認してから残りを回すこと。
セル9（アドホック生成）は、チャプター定義に縛られず素材＋プロンプトから単発で1本作るときに使う。

並列生成: セル1の`WORKERS`でComfyUIを複数プロセス立てて同時生成できる（A100 40GBで`WORKERS=2`が目安。L4は`1`のまま）。
VRAMは`--reserve-vram`でワーカー数に分割される。速くなるのはモデルロード・VAEデコード・IOが重なる分で、
サンプリングは同じGPUの取り合いになるため1本あたりは遅くなる。

課金の注意: CUは「GPUランタイム接続中の時間」で消費される（セル実行中でなくても）。終わったら必ず「ランタイム → ランタイムを接続解除して削除」。


In [ ]:
#@title 0.（初回のみ・無料CPUランタイムでOK）重みをGCS/Driveへ事前配置 — GPUセッションのCU消費とDL待ちをなくす
# 使い方: 「ランタイム → ランタイムのタイプを変更 → CPU」にして、このセルだけを実行する（セル1以降は不要）。
# 配置先に完全な重みが揃っていれば何もしないので、再実行は常に安全。完了後はこのCPUランタイムを削除してよい。
# GCS_BUCKET設定時はGCSへ配置する（GPUセッションのDLがDrive FUSEの83MB/sに対し数百MB/s級になる）。
# Driveに既にある重みはHFから落とし直さず、Drive→GCSへ直接コピーして移行する。
GCS_BUCKET = "auto"      # "auto"=プロジェクトIDから gs://<project-id>-h3-weights を導出 / "gs://名"を直接指定 / ""=GCS不使用（従来のDrive配置）
DRIVE_DIR = "/content/drive/MyDrive/h3_weights"  # Drive配置先。GCS移行の際は既存重みのコピー元としても使う。Drive運用をやめた後は "" にする
TARGET_GPUS = ["L4", "A100"]  # 事前配置する対象GPU。["L4"] / ["A100"] / 両方（両方＝unet4本で約117GB）
INCLUDE_I2V = True       # I2Vチャプター用unet（fl2va）も配置する
INCLUDE_R2V = True       # R2Vチャプター用unet（ref2va）も配置する

import os, shutil, subprocess
REPO = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main"
SIZES = {  # HF上の正確なバイト数（2026-08時点）。完全性チェックに使う
    "vae/minimax_h3_audio_vae_fp32.safetensors": 605_254_808,
    "vae/minimax_h3_video_vae_fp16.safetensors": 5_207_808_496,
    "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors": 27_141_342_152,
    "diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors": 20_958_205_608,
    "diffusion_models/minimax_h3_ref2va_pruned_fp8_scaled.safetensors": 20_958_205_608,
    "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors": 20_970_379_616,
    "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors": 20_970_379_616,
}
UNET_Q = {"L4": "fp8_scaled", "A100": "int8_convrot"}  # Ada=fp8 / Ampere=int8（エンコーダはどちらもint8_convrot）

need = ["vae/minimax_h3_audio_vae_fp32.safetensors",
        "vae/minimax_h3_video_vae_fp16.safetensors",
        "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors"]
for g in TARGET_GPUS:
    q = UNET_Q[g]
    if INCLUDE_I2V:
        need.append(f"diffusion_models/minimax_h3_fl2va_pruned_{q}.safetensors")
    if INCLUDE_R2V:
        need.append(f"diffusion_models/minimax_h3_ref2va_pruned_{q}.safetensors")
need = list(dict.fromkeys(need))

# --- 配置先のセットアップ ---
bucket, gcs_have = None, {}
if GCS_BUCKET:
    from google.colab import auth
    auth.authenticate_user()  # 初回はポップアップ承認（Colabと同じGoogleアカウントを選ぶ）
    proj = subprocess.run(["gcloud", "config", "get-value", "project"],
                          capture_output=True, text=True).stdout.strip()
    if not proj or proj == "(unset)":
        proj = subprocess.run(["gcloud", "projects", "list", "--format=value(projectId)", "--limit=1"],
                              capture_output=True, text=True).stdout.strip()
        assert proj, "GCPプロジェクトが見つからない — 請求先アカウントとプロジェクトを先に作成する"
        subprocess.run(["gcloud", "config", "set", "project", proj], capture_output=True, check=True)
    bucket = (GCS_BUCKET if GCS_BUCKET.startswith("gs://")
              else (f"gs://{proj}-h3-weights" if GCS_BUCKET == "auto" else f"gs://{GCS_BUCKET}"))
    if subprocess.run(["gcloud", "storage", "buckets", "describe", bucket],
                      capture_output=True).returncode != 0:
        print(f"GCSバケットを作成: {bucket}（USマルチリージョン・Standard・project: {proj}）", flush=True)
        _mk = subprocess.run(["gcloud", "storage", "buckets", "create", bucket, "--location=US",
                              "--default-storage-class=STANDARD", "--uniform-bucket-level-access"],
                             capture_output=True, text=True)
        if _mk.returncode != 0:
            _err = (_mk.stderr or _mk.stdout).strip()
            _low = _err.lower()
            if "billing" in _low or "403" in _low:
                _hint = (f"\n→ プロジェクト {proj} に請求先アカウントがリンクされていない可能性が高い。\n"
                         f"  https://console.cloud.google.com/billing/linkedaccount?project={proj}\n"
                         "  で「アカウントをリンク」してから、このセルを再実行する（請求先アカウントの作成と"
                         "プロジェクトへのリンクは別手順）")
            elif "409" in _low or "already" in _low:
                _hint = "\n→ 同名バケットが既に存在する（他人所有の可能性）。GCS_BUCKET に別名を直接指定する"
            else:
                _hint = "\n→ 上のエラーメッセージを確認して対処後、このセルを再実行する"
            raise SystemExit(f"バケット作成に失敗:\n{_err}{_hint}")
    _ls = subprocess.run(["gcloud", "storage", "ls", "-l", bucket + "/"], capture_output=True, text=True)
    for _line in _ls.stdout.splitlines():
        _p = _line.split()
        if len(_p) >= 3 and _p[0].isdigit() and _p[-1].startswith("gs://"):
            gcs_have[os.path.basename(_p[-1])] = int(_p[0])
    print(f"★ GCS配置先: {bucket}（既存 {len(gcs_have)}本）")
drive_dir = DRIVE_DIR or None
if drive_dir:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(drive_dir, exist_ok=True)
assert bucket or drive_dir, "GCS_BUCKET と DRIVE_DIR の少なくとも一方を設定する"
if not shutil.which("aria2c"):
    subprocess.run(["apt-get", "-qq", "-y", "install", "aria2"], check=True, capture_output=True)

TMP = "/content/h3_dl"
os.makedirs(TMP, exist_ok=True)
for rel in need:
    n, sz = os.path.basename(rel), SIZES[rel]
    drv = f"{drive_dir}/{n}" if drive_dir else None
    on_gcs = bool(bucket) and gcs_have.get(n) == sz
    on_drive = bool(drv and os.path.exists(drv) and os.path.getsize(drv) == sz)
    # 目標: GCS運用ならGCSに揃える（Driveへは足さない）。GCS不使用なら従来どおりDriveに揃える
    if (bucket and on_gcs) or (not bucket and on_drive):
        print("OK（配置済み）", n)
        continue
    if bucket and on_drive:
        print(f"=== {n}: Drive → GCS へ直接コピー（FUSE読み≒80MB/s・21GBで約5分） ===", flush=True)
        subprocess.run(["gcloud", "storage", "cp", drv, f"{bucket}/{n}"], check=True)
        print("  gcs OK", n)
        continue
    # どこにも無い → HFからDL
    local = f"{TMP}/{n}"
    free = shutil.disk_usage(TMP).free
    assert free >= sz + 2_000_000_000, f"ローカルディスク不足（空き{free / 2**30:.0f}GiB）— ランタイムを作り直す"
    print(f"=== {n} をHFからDL（aria2・16並列・中断してもセル再実行でレジューム） ===", flush=True)
    r = subprocess.run(["aria2c", "-c", "-x16", "-s16", "--file-allocation=none",
                        "--summary-interval=30", "--console-log-level=warn",
                        "-d", TMP, "-o", n, f"{REPO}/{rel}"])
    assert r.returncode == 0 and os.path.getsize(local) == sz, f"{n} のDL不完全 — このセルを再実行"
    if bucket:
        print(f"  -> GCSへアップロード中: {bucket}/{n}", flush=True)
        subprocess.run(["gcloud", "storage", "cp", local, f"{bucket}/{n}"], check=True)
        print("  gcs OK", n)
    else:
        dst = f"{drive_dir}/{n}"
        free = shutil.disk_usage(drive_dir).free
        assert free >= sz + 2_000_000_000, (
            "Drive空き不足。プランを上げるか、不要ファイル削除＋ゴミ箱を空にする（ゴミ箱も容量にカウントされる）")
        print("  -> Driveへコピー中（FUSE越しで数分〜十数分）", flush=True)
        if os.path.exists(dst):
            os.remove(dst)
        shutil.copy(local, dst)
        assert os.path.getsize(dst) == sz, f"{n} のDriveコピー不完全 — このセルを再実行"
        print("  OK", n)
    os.remove(local)  # ローカルは都度消してディスクを使い回す
if drive_dir:
    from google.colab import drive
    drive.flush_and_unmount()  # Driveへの書き込みを書き切ってから終了（これが済むまでランタイムを消さない）
print("完了。このランタイムは削除してよい。GCS配置なら、以後のセッションはセル1の WEIGHTS_GCS_BUCKET で使われる")

In [ ]:
#@title 1. 設定＋環境チェック（毎セッションここだけ編集。実行すると選択中のGPUと使用重みを表示）
CHAPTERS = []                # 生成するチャプター。空 = バンドル内の全チャプターを番号順に生成。パイロット運用なら ["ch2"] → 合格後 [] （生成済みは自動スキップ）
EXPECTED_GPU = "A100"            # 例 "A100" / "L4"。設定すると、ランタイムの選択がそれと違うときにここで止まる（設定変更漏れの検知）
WORKERS = 2                  # 同時生成数。1=直列（L4はこちら）。2以上は1枚のGPUをVRAM分割してComfyUIを複数立てる（A100 40GBで2が目安）
BUNDLE_ZIP_FROM_DRIVE = "/content/drive/MyDrive/h3_inputs/"   # zipのフルパス、またはzipを置いたディレクトリ（最新zipを使う）。空ならセル4でブラウザからアップロード
WEIGHTS_DRIVE_DIR = "/content/drive/MyDrive/h3_weights"       # 例 "/content/drive/MyDrive/h3_weights"。Driveを重みキャッシュに使う（DL回避。実行前に必要分をローカルへコピーする）。GCSへ移行しDriveの重みを消したら "" にする
WEIGHTS_GCS_BUCKET = "auto"  # GCSを重みキャッシュに使う（Drive比で数倍速いDL。設定時はDriveより優先）。"auto"=プロジェクトIDから gs://<project-id>-h3-weights を導出 / "gs://名"直接指定 / ""=GCS不使用。初回配置はセル0。認証失敗時はDrive/HFへ自動フォールバック
SAVE_WEIGHTS_TO_DRIVE = True # WEIGHTS_DRIVE_DIR設定時、HFから落とした重みをDriveへ保存する（次回セッションが数分で立ち上がる）
OUT_DRIVE_DIR = "/content/drive/MyDrive/h3_outputs/47_fukuchan"           # 例 "/content/drive/MyDrive/h3_outputs"。設定すると各チャプター完了ごとに即Driveへ退避（切断事故に強い）
NEED_I2V = False              # I2Vチャプター（fl2va 21GB）を使う
NEED_R2V = True              # R2Vチャプター（ref2va 21GB）を使う
COMFY_FLAGS = []             # 生成中に CUDA out of memory が出たら ["--lowvram"] にしてセル6から再実行
SAGE_ATTENTION = False       # SageAttention（INT8量子化attention）。2026-08実測: **A100=x0.95で逆効果＝False固定、L4=x1.07で採用＝L4本番はTrueにする**（品質はパイロットで確認）。変更したらセル6の再実行が必要
FAST_FLAGS = []              # ComfyUIの--fast最適化。2026-08実測: **L4・A100とも効果ゼロ（x1.00）＝常用しない**（H3はcomfy-aimdoの量子化カーネル経由で、--fastが差し替える標準経路を通らない）
AB_LABEL = ""                # A/B計測ラベル（例 "base"→"sage"）。設定すると成果物名が ch1__<ラベル>.mp4 になり、生成済みスキップと衝突せずに同一チャプター・同一シードを別条件で再生成できる。所要時間はセル7が bench_log.csv に自動記録。通常運用では空
AUTO_SHUTDOWN = True        # セル7が指定チャプターを全て成功させたら、ランタイムを自動で切断・削除して課金を止める。
                             # OUT_DRIVE_DIR が必須（ランタイムを消すと /content/outputs のmp4は失われるため。
                             # 全チャプターのDrive退避を確認できないときは切断しない）。
                             # パイロット中や、セル8でブラウザに落としたいときは False のままにする
SHUTDOWN_GRACE_SEC = 5      # 自動切断までの猶予秒。この間にセルを停止（■ボタン）すれば切断を取り消せる

# --- 以下は自動判定（編集不要）。このランタイムで実際に選択されているGPUを正として重みを決める ---
import re, shutil, torch, psutil
if torch.cuda.is_available():
    NAME, CAP = torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)
    VRAM = torch.cuda.get_device_properties(0).total_memory / 2**30
else:
    # 無料CPUランタイム: 生成はできないが、セル2→3で重みをDL→Driveへ配置する用途（0円）に使える
    NAME, CAP, VRAM = "CPU（重み配置専用モード — セル3まで実行、生成セルは不可）", (8, 9), 0.0
    print("⚠ GPUなし: 重みのDrive配置専用モードとして続行（L4/Ada向けバリアントを選択）")
if EXPECTED_GPU:
    assert EXPECTED_GPU.lower() in NAME.lower(), (
        f"意図したGPU「{EXPECTED_GPU}」と実際の割当「{NAME}」が違う！\n"
        f"ランタイム → ランタイムのタイプを変更 → {EXPECTED_GPU} を選んで再接続してから、このセルを再実行")
if torch.cuda.is_available():
    assert CAP >= (8, 0), f"{NAME} は生成に使えない（Turing以下）。L4以上のGPUを選ぶ"
assert re.fullmatch(r"[A-Za-z0-9_-]*", AB_LABEL), "AB_LABEL は英数字と - _ のみ（ファイル名になる）"
RAM = psutil.virtual_memory().total / 2**30
DISK = shutil.disk_usage("/content").free / 2**30

if CAP >= (10, 0):        # Blackwell
    ENCODER, FP8 = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", True
elif CAP >= (8, 9):       # Ada (L4/RTX40xx) / Hopper
    ENCODER, FP8 = "qwen3vl_32b_minimax_h3_int8_convrot.safetensors", True
else:                     # Ampere (A100)
    ENCODER, FP8 = "qwen3vl_32b_minimax_h3_int8_convrot.safetensors", False
Q = "fp8_scaled" if FP8 else "int8_convrot"
UNET_I2V = f"minimax_h3_fl2va_pruned_{Q}.safetensors"
UNET_R2V = f"minimax_h3_ref2va_pruned_{Q}.safetensors"

# 並列生成: ComfyUIはプロセスあたり1本ずつしか実行しないので、同時生成にはプロセスを複数立てて
# VRAMを分け合う。各ワーカーの上限は --reserve-vram（「使わずに残す量」）で決める。
assert isinstance(WORKERS, int) and WORKERS >= 1, "WORKERS は1以上の整数"
SERVERS = [f"127.0.0.1:{8188 + i}" for i in range(WORKERS)]
RESERVE_VRAM = round(VRAM - VRAM / WORKERS + 1.0, 1) if WORKERS > 1 else 0.0

# 自動切断は「成果物がDriveに残る」ことが前提。ここで弾いておかないと、
# 生成し終えた直後にmp4ごとランタイムを消してしまう。
if AUTO_SHUTDOWN:
    assert OUT_DRIVE_DIR, (
        "AUTO_SHUTDOWN には OUT_DRIVE_DIR が必要 — ランタイムを削除すると /content/outputs のmp4は消える。"
        " OUT_DRIVE_DIR を設定するか、AUTO_SHUTDOWN=False にしてセル8で回収する")

print(f"★ このセッションのGPU: {NAME} (SM {CAP[0]}.{CAP[1]})  VRAM {VRAM:.1f} GiB  RAM {RAM:.1f} GiB  空きディスク {DISK:.1f} GiB")
print(f"★ 使用する重み: {UNET_I2V} / {UNET_R2V} / {ENCODER}")
if SAGE_ATTENTION or FAST_FLAGS or AB_LABEL:
    print(f"★ 高速化/計測: SAGE_ATTENTION={SAGE_ATTENTION}  FAST_FLAGS={FAST_FLAGS or '(なし)'}  AB_LABEL={AB_LABEL or '(なし)'}")
    if FAST_FLAGS and CAP < (8, 9):
        print("  ⚠ FAST_FLAGS はこのGPUでは効かない（fp8_matrix_multはAda/Hopper専用、fp16_accumulationはA100だと等速）— 外してよい")
    if AB_LABEL:
        print(f"  A/B計測モード: 成果物は ch*__{AB_LABEL}.mp4。計測は /content/outputs/bench_log.csv へ追記"
              "（OUT_DRIVE_DIR設定時はDriveにもコピー）。フラグを変えたらセル1→6→7の順で再実行")
print("★ 設定:", dict(CHAPTERS=CHAPTERS, WORKERS=WORKERS, NEED_I2V=NEED_I2V, NEED_R2V=NEED_R2V,
                     WEIGHTS_DRIVE_DIR=WEIGHTS_DRIVE_DIR or "(未使用)", WEIGHTS_GCS_BUCKET=WEIGHTS_GCS_BUCKET or "(未使用)",
                     OUT_DRIVE_DIR=OUT_DRIVE_DIR or "(未使用)"))
if WORKERS > 1:
    ports = ", ".join(s.split(":")[1] for s in SERVERS)
    print(f"★ 並列生成: {WORKERS}ワーカー（ポート {ports}）／1ワーカーあたりVRAM上限 約{VRAM / WORKERS:.0f} GiB"
          f"（--reserve-vram {RESERVE_VRAM}）")
    print(f"  並列で速くなるのはモデルロード・VAEデコード・IOが重なる分で、サンプリング自体は同じGPUの"
          f"取り合いになる。1本あたりは遅くなり、総時間が縮む形になる")
    print(f"  ComfyUIを{WORKERS}個立てるのでホストRAM {RAM:.0f} GiB も分け合う。ワーカーが突然消える／"
          f"ログに Killed が出たらRAM不足 → WORKERS=1 に戻してセル1→6→7を再実行")
    if VRAM / WORKERS < 18:
        print(f"  ⚠ 1ワーカー {VRAM / WORKERS:.0f} GiB はH3には少ない（L4は実測22 GiBで動作）。"
              f"OOMなら WORKERS を減らすか COMFY_FLAGS=['--lowvram']")
if AUTO_SHUTDOWN:
    print(f"★ 自動切断: 有効 — セル7が全チャプター成功し、その全てが {OUT_DRIVE_DIR} に退避済みなら、"
          f"{SHUTDOWN_GRACE_SEC}秒のカウントダウン後にランタイムを削除して課金を止める")
    print(f"  1本でも失敗／Drive未退避があれば切断しない。カウントダウン中にセルを停止（■）すれば取り消せる")
    print(f"  ⚠ 切断後は /content 以下が全て消える。パイロットの確認やセル8での回収を挟むなら False にすること")
else:
    print("★ 自動切断: 無効 — 生成後もランタイムは接続されたまま課金が続く。"
          "終わったら「ランタイム → ランタイムを接続解除して削除」を忘れずに")

# ディスク見積り: 実行時の重みはローカル実体が必須（Drive FUSE越しのsymlink参照は不可・実測）。
# Drive利用時はユニットを1本ずつ入れ替えるので「エンコーダ27＋VAE6＋ユニット21≒54GB」あれば足りる。
# Drive無しで両ユニットをDLする場合は約75GB必要（L4の実測ディスク65GBには収まらない）。
# 重みのローカル実体はワーカー間で共有する（同じファイルを各プロセスが読む）ので、WORKERSを増やしても増えない。
est = 6 + 27 + (21 if WEIGHTS_DRIVE_DIR else 21 * (NEED_I2V + NEED_R2V))
if DISK < est + 5:
    print(f"⚠ 空き{DISK:.0f}GiBに対し必要見積り約{est}GB — "
          + ("不要ファイルの削除を検討" if WEIGHTS_DRIVE_DIR else "WEIGHTS_DRIVE_DIRの利用か、NEEDフラグを片方ずつにすることを推奨"))

In [ ]:
#@title ★ 一括実行（セル1〜7をまとめて実行 — セル1を編集したら、あとはこのセルだけ押せばよい）
# セル1→2→…→7を1つずつ押す手間を無くすランナー。Colab標準の「すべてのセルを実行」は
# セル0（無料CPU専用の重み事前配置）やセル9（アドホック生成）まで走ってしまうので使えない。
# ここではノートブック自身のセルソースを取り出し、指定した番号のセルだけを上から順に実行する。
# 操作が必要なもの（バンドルzipのアップロード）は最初にまとめて済ませるので、以降は無人で流れる。
#@markdown - `FROM_STEP` / `TO_STEP`: 実行するセル番号の範囲（既定 1〜7）。失敗したセルから再開するときは `FROM_STEP` をその番号にする（生成済みチャプターはセル7がスキップするのでやり直しは安い）
FROM_STEP = 1  #@param {type:"integer"}
TO_STEP = 7  #@param {type:"integer"}
#@markdown - `FAIL_SHUTDOWN_SEC`: **失敗して止まったとき**、この秒数のカウントダウン後にランタイムを切断・削除して課金を止める（`0` = 切断しない）。カウントダウン中にこのセルを停止（■）すれば取り消せる。Driveへ退避されていない成果物があるときは切断しない。切断前にComfyUIログの末尾を出力へ残す（`OUT_DRIVE_DIR`設定時はログもDriveへコピー）
FAIL_SHUTDOWN_SEC = 600  #@param {type:"integer"}
#@markdown ---
#@markdown バンドルzipをブラウザからアップロードする場合（`BUNDLE_ZIP_FROM_DRIVE`が空のとき）は**最初に**求められる。以降は操作不要。

RUN_ALL_CELL_MARKER = True  # このセル自身を実行対象から外すための目印（消さない）
import glob as _ra_glob, os as _ra_os, re as _ra_re, shutil as _ra_shutil, time as _ra_time

def _ra_load_steps():
    # ノートブックの現在の内容（フォームの編集も反映済み）を取り出し、
    # 先頭行の「#@title <番号>.」/「# <番号>.」から番号→ソースの対応を作る。
    from google.colab import _message
    res = _message.blocking_request("get_ipynb", timeout_sec=120)
    nb = res["ipynb"] if isinstance(res, dict) and "ipynb" in res else res
    steps = {}
    for c in nb["cells"]:
        if c.get("cell_type") != "code":
            continue
        src = c["source"]
        src = src if isinstance(src, str) else "".join(src)
        if "RUN_ALL_CELL_MARKER" in src:
            continue
        for line in src.split("\n")[:3]:
            m = _ra_re.match(r"\s*(?:#@title|#)\s*(\d)[.．]", line)
            if m:
                steps.setdefault(int(m.group(1)), src)
                break
    return steps

def _ra_premount():
    # Driveのマウントは初回に承認ダイアログが出ることがある = 操作が必要。アップロードと一緒に
    # 先に済ませておけば、セル3以降のDriveアクセスで止まらない。
    if not any(globals().get(k) for k in ("WEIGHTS_DRIVE_DIR", "BUNDLE_ZIP_FROM_DRIVE", "OUT_DRIVE_DIR")):
        return
    if _ra_os.path.isdir("/content/drive/MyDrive"):
        return
    from google.colab import drive as _gd
    print("★ 先にGoogle Driveをマウントする（初回は承認ダイアログが出る）", flush=True)
    _gd.mount("/content/drive")

def _ra_preupload():
    # 操作が必要なアップロードを先に済ませる（重いインストール・重み配置の前）。
    # ここで置いたパスを BUNDLE_ZIP_LOCAL に入れると、セル4がアップロードを求めずそれを使う。
    if 4 not in _RA_TODO or globals().get("BUNDLE_ZIP_FROM_DRIVE"):
        return
    cur = globals().get("BUNDLE_ZIP_LOCAL") or ""
    if cur and _ra_os.path.exists(cur):
        print(f"  バンドルzipは既にローカルにある（セル4はこれを使う）: {cur}", flush=True)
        return
    from google.colab import files
    print("★ 先にバンドルzip（<NN>_<slug>_bundle.zip）をアップロードする — 操作が必要なのはここだけ",
          flush=True)
    up = files.upload()
    zips = [n for n in up if n.lower().endswith(".zip")]
    assert zips, "zipが選ばれていない — バンドルzipを選ぶか、セル1で BUNDLE_ZIP_FROM_DRIVE を設定する"
    globals()["BUNDLE_ZIP_LOCAL"] = _ra_os.path.join(_ra_os.getcwd(), zips[0])
    print(f"  アップロード完了: {BUNDLE_ZIP_LOCAL} "
          f"({_ra_os.path.getsize(BUNDLE_ZIP_LOCAL) / 2**20:.1f} MB)。以降は無人で流れる", flush=True)

def _ra_fail_shutdown():
    # 失敗して止まったあと放置されると、GPUランタイムが繋がっている間ずっとCUを食う。
    # 診断材料（ComfyUIログ末尾）を出力に残してから、猶予後にランタイムを削除して課金を止める。
    outdir = globals().get("OUT_DRIVE_DIR") or ""
    for lg in sorted(_ra_glob.glob("/content/comfyui_*.log")):
        try:
            with open(lg, errors="replace") as fo:
                tail = fo.read()[-4000:]
            print(f"\n--- {lg}（末尾） ---\n{tail}", flush=True)
            if outdir and _ra_os.path.isdir(outdir):
                _ra_shutil.copy(lg, outdir)
                print(f"--- {lg} をDriveへ保存: {outdir}", flush=True)
        except OSError as e:
            print(f"--- {lg} を読めなかった: {e}", flush=True)
    if FAIL_SHUTDOWN_SEC <= 0:
        print("（FAIL_SHUTDOWN_SEC=0 のため自動切断しない。放置すると課金が続くので、"
              "終わったら「ランタイム → ランタイムを接続解除して削除」）", flush=True)
        return
    outs = sorted(_ra_glob.glob("/content/outputs/*.mp4"))
    unsaved = [_ra_os.path.basename(o) for o in outs
               if not (outdir and _ra_os.path.exists(_ra_os.path.join(outdir, _ra_os.path.basename(o))))]
    if unsaved:
        print(f"⚠ 自動切断を中止: Driveに退避されていない成果物がある {unsaved} —"
              " セル8で回収してから手動でランタイムを削除すること", flush=True)
        return
    print(f"★ {FAIL_SHUTDOWN_SEC}秒後にランタイムを切断・削除して課金を止める。"
          "取り消すならこのセルを停止（■）", flush=True)
    left, step = FAIL_SHUTDOWN_SEC, max(15, FAIL_SHUTDOWN_SEC // 10)
    while left > 0:
        print(f"   切断まで {left}秒...", flush=True)
        _ra_time.sleep(min(step, left))
        left -= min(step, left)
    try:
        from google.colab import drive as _gd
        _gd.flush_and_unmount()  # Driveへの書き込み（成果物・ログ）を確実に反映させてから消す
        print("   Driveへ書き切った", flush=True)
    except Exception as e:
        print(f"   ⚠ flush_and_unmount に失敗: {e}", flush=True)
    print("   ランタイムを削除する。以降の出力は表示されない", flush=True)
    from google.colab import runtime
    runtime.unassign()

try:
    _RA_STEPS = _ra_load_steps()
except Exception as _ra_e:
    raise SystemExit(f"この環境ではノートブックのセルを取り出せなかった（{type(_ra_e).__name__}: {_ra_e}）"
                     " — セル1〜7を順に手で実行すること")
# 立ち上げの並列化: セル3（重み配置）は呼ぶとバックグラウンドスレッドでコピー／DLを進めるので、
# 番号順（2→3）ではなく先に始める。33〜54GBのコピーがセル2のpip install・セル4のバンドル展開と
# 並走し、その分だけ全体が短くなる（ComfyUIを起動するセル6より前にセル2が終わっていればよい）。
RUN_ORDER = [1, 3, 4, 2, 5, 6, 7]
_RA_RANGE = list(range(FROM_STEP, TO_STEP + 1))
_RA_TODO = [n for n in RUN_ORDER if n in _RA_RANGE]
_RA_SKIP = [n for n in _RA_RANGE if n not in RUN_ORDER]
if _RA_SKIP:
    print(f"（セル{_RA_SKIP} は一括実行の対象外 — セル8=回収・セル9=単発生成は手で実行する）", flush=True)
_RA_MISSING = [n for n in _RA_TODO if n not in _RA_STEPS]
assert not _RA_MISSING, (f"セル{_RA_MISSING} が見つからない — 各セル先頭の「#@title <番号>.」を"
                         "書き換えると検出できなくなる（このセル自身は番号を持たない）")
if FROM_STEP > 1 and "SERVERS" not in globals():
    print("⚠ セル1をこのセッションで実行していない（設定が未定義）— FROM_STEP=1 から流すこと", flush=True)

_ra_ip = get_ipython()
_ra_t0 = _ra_time.time()
_ra_err = None
print(f"★ 一括実行: セル{' → '.join(map(str, _RA_TODO))}（{len(_RA_TODO)}本）"
      "／番号順でないのは、セル3の重みコピーをセル2のpip installと並走させるため", flush=True)
try:
    if FROM_STEP > 1:  # セル1を再実行しない再開時も、必要な操作は先に済ませる
        _ra_premount()
        _ra_preupload()
    for _ra_n in _RA_TODO:
        print(f"\n{'=' * 78}\n▶ セル{_ra_n} 開始（経過 {(_ra_time.time() - _ra_t0) / 60:.1f}分）\n{'=' * 78}",
              flush=True)
        _ra_res = _ra_ip.run_cell(_RA_STEPS[_ra_n])
        if not _ra_res.success:
            raise RuntimeError(f"セル{_ra_n} で失敗（原因は直前のトレースバック）。直したら "
                               f"FROM_STEP={_ra_n} にしてこのセルを再実行すれば続きから流せる")
        print(f"✔ セル{_ra_n} 完了（経過 {(_ra_time.time() - _ra_t0) / 60:.1f}分）", flush=True)
        if _ra_n == 1:
            # 設定が読めた直後 = 重いセル2〜4の前に、操作が必要なものをまとめて済ませる
            _ra_premount()
            _ra_preupload()
except KeyboardInterrupt:
    print("\n■ 停止された（自動切断はしない）。課金を止めるなら「ランタイム → 接続解除して削除」", flush=True)
    raise
except BaseException as _ra_e2:
    _ra_err = _ra_e2

if _ra_err is None:
    print(f"\n★ セル{FROM_STEP}〜{TO_STEP} 完了（合計 {(_ra_time.time() - _ra_t0) / 60:.1f}分）"
          " — 成果物の回収はセル8、単発生成はセル9", flush=True)
else:
    print(f"\n■ 一括実行を中断: {_ra_err}", flush=True)
    _ra_fail_shutdown()
    raise SystemExit(str(_ra_err))

In [ ]:
%%bash
# 2. ComfyUIインストール＋aria2導入（2〜3分）
set -e
apt-get -yq install aria2 > /dev/null 2>&1 || true
cd /content
# 一括実行では重み配置（セル3）・バンドル投入（セル4）を先に走らせて pip install と並走させるため、
# /content/ComfyUI が「models/ や input/ だけ先に作られた状態」で来ることがある。ディレクトリの
# 有無で判定すると clone が丸ごとスキップされてしまうので、main.py の有無で判定し、別の場所へ
# cloneしてから既存ファイルを上書きせずマージする（cp -n）。
if [ ! -f ComfyUI/main.py ]; then
  rm -rf /content/ComfyUI_clone
  git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI_clone
  mkdir -p /content/ComfyUI
  cp -rn /content/ComfyUI_clone/. /content/ComfyUI/
  rm -rf /content/ComfyUI_clone
fi
cd ComfyUI
pip install -q -r requirements.txt
test -f comfy_extras/nodes_minimax_h3.py && echo "MiniMax H3 nodes: OK" \
  || { echo "ERROR: nodes_minimax_h3.py が無い — ComfyUIが古い"; exit 1; }


In [ ]:
#@title 3. 重み配置（バックグラウンド実行 — 開始したらそのままセル4〜6へ進んでよい。セル7が完了を待つ）
# GPUごとに使うユニット重みが違う（L4/Ada=fp8ペア、A100/Ampere=int8ペア。各21GB×2）。
# エンコーダとVAE（約33GB）は共通。Driveには「共通分＋今のGPUのペア」だけを置く方針で、
# 必要な重みがDriveに無く、代わりに別バリアントがある場合は削除してから新しい方をDL＆保存する
# （共通33GB＋ペア42GB≒75GBで、Google One 100GBに常に収まる）。
import concurrent.futures, glob, os, shutil, subprocess, threading
REPO = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main"
SUB = lambda n: "vae" if "_vae_" in n else ("text_encoders" if n.startswith("qwen3vl") else "diffusion_models")
ALT_VARIANTS = {  # 同じモデルの別量子化名（GPU切替時にDriveから退避する対象）
    "minimax_h3_fl2va_pruned_fp8_scaled.safetensors": ["minimax_h3_fl2va_pruned_int8_convrot.safetensors"],
    "minimax_h3_fl2va_pruned_int8_convrot.safetensors": ["minimax_h3_fl2va_pruned_fp8_scaled.safetensors"],
    "minimax_h3_ref2va_pruned_fp8_scaled.safetensors": ["minimax_h3_ref2va_pruned_int8_convrot.safetensors"],
    "minimax_h3_ref2va_pruned_int8_convrot.safetensors": ["minimax_h3_ref2va_pruned_fp8_scaled.safetensors"],
    "qwen3vl_32b_minimax_h3_int8_convrot.safetensors": ["qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"],
    "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": ["qwen3vl_32b_minimax_h3_int8_convrot.safetensors"],
}
MIN_BYTES = {  # 不完全ファイル検出用の下限（実サイズの少し下）
    "minimax_h3_video_vae_fp16.safetensors": 5_000_000_000,
    "minimax_h3_audio_vae_fp32.safetensors": 550_000_000,
    "qwen3vl_32b_minimax_h3_int8_convrot.safetensors": 26_000_000_000,
    "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": 15_000_000_000,
    "minimax_h3_fl2va_pruned_fp8_scaled.safetensors": 20_900_000_000,
    "minimax_h3_ref2va_pruned_fp8_scaled.safetensors": 20_900_000_000,
    "minimax_h3_fl2va_pruned_int8_convrot.safetensors": 20_000_000_000,
    "minimax_h3_ref2va_pruned_int8_convrot.safetensors": 20_000_000_000,
}
need = ["minimax_h3_video_vae_fp16.safetensors", "minimax_h3_audio_vae_fp32.safetensors", ENCODER]
if NEED_I2V: need.append(UNET_I2V)
if NEED_R2V: need.append(UNET_R2V)
UNETS_NEEDED = [u for u, f in ((UNET_I2V, NEED_I2V), (UNET_R2V, NEED_R2V)) if f]

if WEIGHTS_DRIVE_DIR or BUNDLE_ZIP_FROM_DRIVE or OUT_DRIVE_DIR:
    from google.colab import drive as _gd
    _gd.mount("/content/drive")  # OAuthが出ることがあるのでここだけ前面で実行
drive_dir = WEIGHTS_DRIVE_DIR or None
if drive_dir:
    os.makedirs(drive_dir, exist_ok=True)

def ok_size(path, n):
    return os.path.exists(path) and os.path.getsize(path) >= MIN_BYTES.get(n, 1)

DRIVE_IO_LOCK = threading.Lock()  # キャッシュ掃除のunmountと、他セルのDrive読み出しの衝突防止

# --- GCS重みキャッシュ（設定時はDriveより優先。gcloud storageの並列DLでFUSEより大幅に速い） ---
def _gcs_setup():
    cfg = str(globals().get("WEIGHTS_GCS_BUCKET") or "").strip()
    if not cfg:
        return None
    try:
        from google.colab import auth as _auth
        _auth.authenticate_user()  # 初回はポップアップ承認（Driveマウントと同様、前面で実行）
        proj = subprocess.run(["gcloud", "config", "get-value", "project"],
                              capture_output=True, text=True).stdout.strip()
        if not proj or proj == "(unset)":
            proj = subprocess.run(["gcloud", "projects", "list", "--format=value(projectId)",
                                   "--limit=1"], capture_output=True, text=True).stdout.strip()
            assert proj, "GCPプロジェクトが見つからない（請求先アカウントとプロジェクトを先に作成する）"
            subprocess.run(["gcloud", "config", "set", "project", proj], capture_output=True, check=True)
        b = (cfg if cfg.startswith("gs://")
             else (f"gs://{proj}-h3-weights" if cfg == "auto" else f"gs://{cfg}"))
        print(f"★ GCS重みキャッシュ: {b}（project: {proj}）", flush=True)
        return b
    except BaseException as e:
        print(f"⚠ GCSを使えない（{e}）— Drive/HFのみで続行", flush=True)
        return None

GCS_BUCKET = _gcs_setup()
GCS_SIZES = {}
if GCS_BUCKET:
    _ls = subprocess.run(["gcloud", "storage", "ls", "-l", GCS_BUCKET + "/"],
                         capture_output=True, text=True)
    for _line in _ls.stdout.splitlines():
        _p = _line.split()
        if len(_p) >= 3 and _p[0].isdigit() and _p[-1].startswith("gs://"):
            GCS_SIZES[os.path.basename(_p[-1])] = int(_p[0])
    print(f"  GCS上の重み: {len(GCS_SIZES)}本"
          + ("" if GCS_SIZES else "（空 — セル0のGCS配置を実行すると以後のセッションが速くなる）"), flush=True)

def gcs_fetch(n, dst):
    # GCS→ローカルDL（成功でTrue）。ディスクが足りなければreclaim_diskで空けてから落とす
    if not GCS_BUCKET or GCS_SIZES.get(n, 0) < MIN_BYTES.get(n, 1):
        return False
    want = headroom_for(GCS_SIZES[n])
    free = reclaim_disk(want, keep=(dst,))
    assert free >= want, disk_short_msg(want, GCS_SIZES[n])
    if os.path.lexists(dst):
        os.remove(dst)  # 旧symlinkが残っているとFUSE越しに書いてしまうため先に消す
    print(f"GCSからDL中（gcloud storage・並列DL）: {n}", flush=True)
    r = subprocess.run(["gcloud", "storage", "cp", f"{GCS_BUCKET}/{n}", dst])
    if r.returncode == 0 and ok_size(dst, n):
        return True
    print(f"  ⚠ GCSからのDLに失敗（rc={r.returncode}）— Drive/HFへフォールバック", flush=True)
    if os.path.exists(dst) and not ok_size(dst, n):
        os.remove(dst)
    return False

def gcs_save(n, src):
    # 重みをGCSへ保存（次回以降の高速化用）。失敗しても生成は止めない
    if not GCS_BUCKET or GCS_SIZES.get(n, 0) >= MIN_BYTES.get(n, 1):
        return
    try:
        if subprocess.run(["gcloud", "storage", "buckets", "describe", GCS_BUCKET],
                          capture_output=True).returncode != 0:
            print(f"  GCSバケットを作成: {GCS_BUCKET}（USマルチリージョン・Standard）", flush=True)
            _mk = subprocess.run(["gcloud", "storage", "buckets", "create", GCS_BUCKET, "--location=US",
                                  "--default-storage-class=STANDARD", "--uniform-bucket-level-access"],
                                 capture_output=True, text=True)
            assert _mk.returncode == 0, ((_mk.stderr or _mk.stdout).strip()
                                         + " — billing系エラーなら請求先アカウントのプロジェクトへのリンクを確認")
        print(f"  -> GCSへ保存中（次回以降の高速化用）: {GCS_BUCKET}/{n}", flush=True)
        subprocess.run(["gcloud", "storage", "cp", src, f"{GCS_BUCKET}/{n}"], check=True)
        GCS_SIZES[n] = os.path.getsize(src)
    except BaseException as e:
        print(f"  ⚠ GCS保存に失敗（生成には影響なし）: {e}", flush=True)

# --- ディスク管理ヘルパ（逼迫時の自動掃除と、不足時に原因が分かる診断） ---
GiB = 2**30
COPY_FLOOR = 2 * GiB   # 大物コピー中に残しておきたい空き（DriveFSの読み出しキャッシュ用ヘッドルーム）
DIFF_DIR = "/content/ComfyUI/models/diffusion_models"
DRIVEFS_CACHE_DIRS = ["/root/.config/Google/DriveFS", "/root/.cache/Google/DriveFS"]

def headroom_for(remaining):
    # コピーに必要な空き = 残りバイト ＋ ヘッドルーム。ヘッドルームは残りに応じて縮める
    # （残り1GiBのレジューム時に「4GiB空いていないから不足」と落ちるのを防ぐ）。
    return remaining + min(COPY_FLOOR, max(remaining // 4, 512 * 2**20))

def free_bytes():
    return shutil.disk_usage("/content").free

def path_bytes(p):
    if os.path.isfile(p) and not os.path.islink(p):
        return os.path.getsize(p)
    total = 0
    for root, _d, fs in os.walk(p, onerror=lambda e: None):
        for f in fs:
            try:
                total += os.lstat(os.path.join(root, f)).st_size
            except OSError:
                pass
    return total

def deleted_open_bytes():
    # 削除済みなのにプロセスが開いたままのファイルは、dfの空きとして戻ってこない（ComfyUIが
    # unetをmmapしたまま等）。「消したのに空きが増えない」ときの原因を特定するために合計を見る。
    total, seen = 0, set()
    for fd in glob.glob("/proc/[0-9]*/fd/*"):
        try:
            if not os.readlink(fd).endswith(" (deleted)"):
                continue
            st = os.stat(fd)
        except OSError:
            continue
        if (st.st_dev, st.st_ino) in seen:
            continue
        seen.add((st.st_dev, st.st_ino))
        total += st.st_size
    return total

def biggest_files(n=6, roots=("/content", "/root"), floor=512 * 2**20):
    hits = []
    for r in roots:
        for root, dirs, fs in os.walk(r, onerror=lambda e: None):
            dirs[:] = [d for d in dirs if os.path.join(root, d) != "/content/drive"]  # Drive側は数えない
            for f in fs:
                p = os.path.join(root, f)
                try:
                    sz = os.lstat(p).st_size
                except OSError:
                    continue
                if sz >= floor:
                    hits.append((sz, p))
    return sorted(hits, reverse=True)[:n]

def disk_short_msg(need, remaining=None):
    # 「空きは残っているのにエラー」を避けるため、判断に使った数字をそのまま出す。
    du = shutil.disk_usage("/content")
    msg = (f"ディスク不足: 空き {du.free / GiB:.1f} GiB / 全体 {du.total / GiB:.1f} GiB、"
           f"必要 {need / GiB:.1f} GiB")
    if remaining is not None:
        msg += f"（コピー残り {remaining / GiB:.1f} GiB ＋ 作業用ヘッドルーム）"
    dob = deleted_open_bytes()
    if dob >= GiB:
        msg += (f"\n  → 削除済みなのにプロセスが掴んでいるファイルが {dob / GiB:.1f} GiB ある"
                "（ComfyUIがunetをmmapしたまま等）。セル6を再実行してComfyUIを再起動すれば解放される")
    big = biggest_files()
    if big:
        msg += "\n  → ローカルの大きいファイル: " + ", ".join(f"{p}={s / GiB:.1f}GiB" for s, p in big)
    return msg + "\n  → 不要ファイルを整理してこのセルを再実行（コピーは途中から再開する）"

def purge_drivefs_cache():
    # DriveFSはFUSE読み出しの内容キャッシュをローカルディスクにも書くため、大物コピー中に
    # 「コピー先＋キャッシュ」の二重消費でディスクが枯渇することがある（ENOSPC実測・2026-08）。
    # unmount→キャッシュ削除→remountで空ける。掃除で実際に何GiB空いたかを必ず表示する
    # （0GiBなら原因はキャッシュではない＝重み本体か、プロセスが掴んだ削除済みファイル）。
    from google.colab import drive as _gd
    lock = globals().get("DRIVE_IO_LOCK")
    if lock:
        lock.acquire()
    try:
        before = free_bytes()
        hit = [d for d in DRIVEFS_CACHE_DIRS if os.path.isdir(d)]
        print(f"  空きディスク逼迫（空き {before / GiB:.1f} GiB）→ DriveFSキャッシュを掃除"
              f"（unmount→削除→remount・数十秒）: {', '.join(hit) or '(キャッシュ無し)'}", flush=True)
        _gd.flush_and_unmount()
        for d in hit:
            shutil.rmtree(d, ignore_errors=True)
        _gd.mount("/content/drive")
        after = free_bytes()
        print(f"  キャッシュ掃除で {max(0, after - before) / GiB:.1f} GiB 解放"
              f"（空き {after / GiB:.1f} GiB）", flush=True)
        return after
    finally:
        if lock:
            lock.release()

def reclaim_disk(need, keep=()):
    # needバイトの空きを作る。安いものから順に捨て、都度measureし直して足りたら止める
    # （1手が効かなかったときに次の手へ進めるようにする）。戻り値は最終的な空きバイト。
    free = free_bytes()
    if free >= need:
        return free
    # 1) 他のunetのローカル実体（Driveに実体があるので消してよい。symlinkは実体を持たないので対象外）
    for other in sorted(glob.glob(f"{DIFF_DIR}/*.safetensors") + glob.glob(f"{DIFF_DIR}/*.part")):
        if free >= need:
            return free
        if other in keep or os.path.islink(other):
            continue
        sz = path_bytes(other)
        os.remove(other)
        print(f"  ディスク確保のためローカルunetを削除（Driveに実体あり）: "
              f"{os.path.basename(other)} -{sz / GiB:.1f} GiB", flush=True)
        free = free_bytes()
    # 2) 生成に不要なキャッシュ・作業ファイル
    junk_list = ["/root/.cache/pip", "/root/.cache/huggingface", "/root/.cache/torch",
                 "/content/h3_dl", "/content/ComfyUI/temp"]
    if glob.glob("/content/bundle/**/script.md", recursive=True):  # 展開済みならバンドルzipは不要
        junk_list += [z for z in sorted(glob.glob("/content/*.zip"))
                      if os.path.basename(z) != "h3_outputs.zip"]  # 回収用zipは消さない
    for junk in junk_list:
        if free >= need:
            return free
        if junk in keep or not os.path.exists(junk):
            continue
        sz = path_bytes(junk)
        if sz < 200 * 2**20:
            continue
        if os.path.isdir(junk) and not os.path.islink(junk):
            shutil.rmtree(junk, ignore_errors=True)
        else:
            os.remove(junk)
        print(f"  ディスク確保のため削除: {junk} -{sz / GiB:.1f} GiB", flush=True)
        free = free_bytes()
    # 3) DriveFSの読み出しキャッシュ（remountに数十秒かかるので最後）
    if free < need:
        free = purge_drivefs_cache()
    return free

def copy_from_drive(src, dst, threads=8, chunk=64 * 2**20):
    # Drive→ローカルのレジューム可能コピー。並列pread（8スレッド・実測62→83MB/s）で読み、
    # 追記順を守って書く＝dstのファイルサイズがそのままレジューム点になる。
    # 空きが逼迫したらキャッシュ掃除等で空けて続行する（shutil.copyだとENOSPCで落ちる）。
    size = os.path.getsize(src)
    done = os.path.getsize(dst) if os.path.exists(dst) else 0

    def read_round(start):  # startからthreads*chunk分を並列preadで読んで返す
        n = min(threads * chunk, size - start)
        def one(i):
            off, ln = start + i * chunk, min(chunk, n - i * chunk)
            if ln <= 0:
                return b""
            fd = os.open(src, os.O_RDONLY)
            try:
                parts, got = [], 0
                while got < ln:
                    b = os.pread(fd, ln - got, off + got)
                    assert b, f"{src} の読み出しが途切れた — セルの再実行で続きから再開する"
                    parts.append(b)
                    got += len(b)
                return b"".join(parts)
            finally:
                os.close(fd)
        with concurrent.futures.ThreadPoolExecutor(threads) as ex:
            return b"".join(ex.map(one, range(threads)))

    with concurrent.futures.ThreadPoolExecutor(1) as ahead:
        nxt = None  # 先読み: 書き込みと次ラウンドの読みを重ねる
        while done < size:
            buf = nxt.result() if nxt else None
            nxt = None
            # 必要な空きは「残りのバイト＋ヘッドルーム」。残りが少ないときにヘッドルームを
            # 理由に止めない（空きが残っているのに落ちるのを防ぐ）。
            want = headroom_for(size - done)
            if free_bytes() < want:
                free = reclaim_disk(want, keep=(dst,))
                assert free >= want, disk_short_msg(want, size - done)
            if buf is None:
                buf = read_round(done)
            if done + len(buf) < size:
                nxt = ahead.submit(read_round, done + len(buf))
            with open(dst, "ab") as fo:
                fo.write(buf)
            if done // (4 * 2**30) != (done + len(buf)) // (4 * 2**30):
                print(f"    ... {(done + len(buf)) / 2**30:.0f}/{size / 2**30:.0f} GiB", flush=True)
            done += len(buf)

def _place_weights():
    for n in need:
        dst = f"/content/ComfyUI/models/{SUB(n)}/{n}"
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if ok_size(dst, n):
            print("skip（配置済み）", n)
            continue
        drv = f"{drive_dir}/{n}" if drive_dir else None
        on_gcs = bool(GCS_BUCKET) and GCS_SIZES.get(n, 0) >= MIN_BYTES.get(n, 1)
        on_drive = bool(drv and ok_size(drv, n))
        # 注意: Drive(FUSE)越しの直接参照symlinkは実行時に使えない（comfy_aimdoの
        # read_file_sliceがFUSE上で失敗する — 2026-08実測）。GCS/Driveはダウンロード回避の
        # キャッシュとして使い、実行前に必要な重みをローカルへコピーする。
        if SUB(n) == "diffusion_models" and len(UNETS_NEEDED) > 1 and (on_gcs or on_drive):
            # ユニット2本はディスクに同時に載らないため、セル7がチャプター毎に使用直前に
            # GCS/Driveからローカル化する（GCSがあればそちら優先）。
            if os.path.lexists(dst):
                os.remove(dst)
            if on_drive and not on_gcs:
                os.symlink(drv, dst)
            print("キャッシュ済み（実体はセル7で使用直前にローカル化）", n)
            continue
        if on_gcs and gcs_fetch(n, dst):
            print("gcs OK（ローカル化済み）", n)
            continue
        if on_drive:
            # 使うユニットが1本だけなら「共通33GB＋ユニット21GB≒54GB」でディスクに収まるので、
            # ここで先にローカル化する。この配置はバックグラウンドで走るため、コピーが
            # セル2のpip install・セル4のバンドル展開・セル6のComfyUI起動と並走する。
            if os.path.lexists(dst):
                os.remove(dst)
            print(f"Driveからローカル化中（8スレッド並列コピー。中断してもレジューム可）: {n}", flush=True)
            copy_from_drive(drv, dst + ".part")
            os.replace(dst + ".part", dst)
            print("drive OK（ローカル化済み）", n)
            if GCS_BUCKET:
                gcs_save(n, dst)  # GCS移行中: Driveにしか無い重みは通りがかりにGCSへも複製する
            continue
        if drv and SAVE_WEIGHTS_TO_DRIVE and not GCS_BUCKET:
            # GPU切替（Drive運用時のみ）: これから保存する分の容量がDriveに足りるなら旧バリアントは
            # 保持する（両方あれば次の切替コストがゼロになる）。足りない場合のみ削除して空ける。
            # 注意: Driveの削除はゴミ箱行きで、ゴミ箱の中身も容量にカウントされる。削除しても保存が
            # 容量不足で失敗する場合は https://drive.google.com でゴミ箱を空にする（保存失敗しても生成は続行される）。
            need_bytes = MIN_BYTES.get(n, 0) + 2_000_000_000
            for alt in ALT_VARIANTS.get(n, []):
                alt_path = f"{drive_dir}/{alt}"
                if not os.path.exists(alt_path):
                    continue
                if shutil.disk_usage(drive_dir).free >= need_bytes:
                    print(f"  Drive容量に余裕があるため別バリアントを保持: {alt}")
                    continue
                os.remove(alt_path)
                print(f"  Drive容量確保のため旧バリアントを削除（ゴミ箱行き）: {alt}")
        print(f"=== {n} をaria2でDL（16並列・15秒毎に進捗表示・中断してもレジューム可） ===", flush=True)
        r = subprocess.run(["aria2c", "-c", "-x16", "-s16", "--file-allocation=none",
                            "--summary-interval=15", "--console-log-level=warn",
                            "-d", os.path.dirname(dst), "-o", n, f"{REPO}/{SUB(n)}/{n}"])
        assert r.returncode == 0 and ok_size(dst, n), f"{n} のDLに失敗 — このセルを再実行すれば途中から再開する"
        print("hf OK", n)
        if GCS_BUCKET:
            gcs_save(n, dst)
        elif drv and SAVE_WEIGHTS_TO_DRIVE:
            try:
                print(f"  -> Driveへ保存中（FUSE越しで時間がかかる。次回以降の高速化用）: {drv}", flush=True)
                shutil.copy(dst, drv)
            except OSError as e:  # Drive容量・一時キャッシュ枯渇などでも生成は止めない
                print(f"  ⚠ Drive保存に失敗（生成には影響なし。後で無料CPUセッションでの配置を推奨）: {e}")
                if os.path.exists(drv):
                    os.remove(drv)
    print(f"★ 重み配置 完了。空きディスク: {shutil.disk_usage('/content').free / 2**30:.1f} GiB", flush=True)

# 配置はバックグラウンドで実行し、ComfyUIインストール（セル2実行済み）後の
# バンドル投入（セル4）〜ComfyUI起動（セル6）と並行させる。完了はセル7の冒頭で待つ。
if "WEIGHTS_THREAD" in globals() and WEIGHTS_THREAD.is_alive():
    print("前回の重み配置がまだ実行中 — 完了を待ってから続行")
    WEIGHTS_THREAD.join()
WEIGHTS_ERR = []

def _bg_place():
    try:
        _place_weights()
    except BaseException as e:
        WEIGHTS_ERR.append(e)
        print(f"⚠ 重み配置が失敗: {e} — セル3を再実行（コピー/DLは途中から再開する）", flush=True)

def wait_weights():
    if WEIGHTS_THREAD.is_alive():
        print("重み配置（バックグラウンド）の完了を待機中...", flush=True)
    WEIGHTS_THREAD.join()
    assert not WEIGHTS_ERR, f"重み配置が失敗している: {WEIGHTS_ERR[0]} — セル3を再実行"

WEIGHTS_THREAD = threading.Thread(target=_bg_place, daemon=True)
WEIGHTS_THREAD.start()
print("★ 重み配置をバックグラウンドで開始 — このままセル4〜6を進めてOK（セル7の冒頭で完了を待つ）")

In [ ]:
#@title 4. バンドル投入（zipをアップロード or Driveから）→ ComfyUI/input/ へ配備
import glob, os, shutil, threading, zipfile

if BUNDLE_ZIP_FROM_DRIVE:
    # セル3のバックグラウンド重み配置がDriveFSキャッシュ掃除（unmount）をすることがあるため、
    # Driveからの読み出しはロックを取ってローカルへ写してから使う
    with globals().get("DRIVE_IO_LOCK") or threading.Lock():
        src = BUNDLE_ZIP_FROM_DRIVE
        if os.path.isdir(src):  # ディレクトリ指定なら中の最新zipを使う
            zips = sorted(glob.glob(os.path.join(src, "*.zip")), key=os.path.getmtime)
            assert zips, f"{src} に*.zipが無い — バンドルzipを置くか、zipのフルパスを指定する"
            src = zips[-1]
            print("ディレクトリ指定: 最新のzipを使う →", src)
        assert os.path.exists(src), f"{src} が無い"
        zp = "/content/" + os.path.basename(src)
        shutil.copy(src, zp)
elif globals().get("BUNDLE_ZIP_LOCAL") and os.path.exists(BUNDLE_ZIP_LOCAL):
    zp = BUNDLE_ZIP_LOCAL  # ★一括実行セルが最初にアップロードしておいたzip（以降を無人で流すため）
    print("アップロード済みのzipを使う:", zp)
else:
    from google.colab import files
    print("バンドルzip（<NN>_<slug>_bundle.zip）を選択:")
    up = files.upload()
    zp = os.path.join(os.getcwd(), next(iter(up)))

shutil.rmtree("/content/bundle", ignore_errors=True)
zipfile.ZipFile(zp).extractall("/content/bundle")
hits = glob.glob("/content/bundle/**/script.md", recursive=True)
assert hits, "zip内にscript.mdが見つからない — ラン専用ディレクトリごとzipしたか確認"
BUNDLE = os.path.dirname(hits[0])
print("BUNDLE =", BUNDLE)

inp = "/content/ComfyUI/input"
os.makedirs(inp, exist_ok=True)
n = 0
for p in sorted(glob.glob(f"{BUNDLE}/*.png") + glob.glob(f"{BUNDLE}/*.wav")):
    if os.path.basename(p).startswith("ref_canvas_"):
        continue
    shutil.copy(p, inp)
    n += 1
print(f"{n} files -> ComfyUI/input/")

In [ ]:
#@title 5. workflowをこのGPUの重み名に調整（SaveVideoのcodec補完込み）
import glob, json, os
wfs = sorted(glob.glob(f"{BUNDLE}/ch*_workflow.json"))
assert wfs, f"{BUNDLE} に ch*_workflow.json が無い — バンドル作成時にworkflowを生成したか確認"
for wf in wfs:
    with open(wf) as f:
        d = json.load(f)
    for node in d.values():
        ins = node.get("inputs", {})
        for k, v in ins.items():
            if not isinstance(v, str):
                continue
            if v.startswith("minimax_h3_fl2va"):
                ins[k] = UNET_I2V
            elif v.startswith("minimax_h3_ref2va"):
                ins[k] = UNET_R2V
            elif v.startswith("qwen3vl_32b"):
                ins[k] = ENCODER
        if node.get("class_type") == "SaveVideo":  # ComfyUI新版(2026-08〜)はcodec必須
            ins.setdefault("codec", "auto")
            ins.setdefault("format", "auto")
    with open(wf, "w") as f:
        json.dump(d, f, indent=1)
    print("adjusted", os.path.basename(wf))
print("重み:", UNET_I2V, "/", UNET_R2V, "/", ENCODER)


In [ ]:
#@title 6. ComfyUI起動（WORKERS個を別ポートで並列起動。プロセスが落ちたらこのセルを再実行）
import json, subprocess, sys, time, urllib.request
SAGE = bool(globals().get("SAGE_ATTENTION"))
FASTF = list(globals().get("FAST_FLAGS") or [])
if SAGE:
    print("SageAttention をインストール中（pip・数十秒）...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sageattention"], check=True)
# 高速化フラグはComfyUIの起動引数なので、セル1で変更したらこのセルの再実行（再起動）が必要
EXTRA_FLAGS = (["--use-sage-attention"] if SAGE else []) + (["--fast", *FASTF] if FASTF else [])
subprocess.run(["pkill", "-f", "main.py --listen"], check=False)
time.sleep(2)
PROCS, LOGS = [], []
for i, srv in enumerate(SERVERS):
    port = srv.split(":")[1]
    # --reserve-vram は「OS/他プロセスのために使わずに残すVRAM量(GB)」。ワーカー数で割った分だけ
    # 使うよう各プロセスに残量を申告させることで、1枚のGPUを取り合っても互いにOOMさせない。
    # 並列時はVRAM上限に加えてsqlite DBもワーカー毎に分ける（同一DBだと "Could not acquire lock
    # on database" / "table alembic_version already exists" の初期化競合ERRORが出る・2026-08実測）
    flags = list(COMFY_FLAGS) + EXTRA_FLAGS + (
        ["--reserve-vram", str(RESERVE_VRAM), "--database-url", f"sqlite:////content/comfyui_db_{i}.sqlite"]
        if WORKERS > 1 else [])
    log = f"/content/comfyui_{i}.log"
    LOGS.append(log)
    PROCS.append(subprocess.Popen(
        [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", port, *flags],
        cwd="/content/ComfyUI", stdout=open(log, "w"), stderr=subprocess.STDOUT))
    print(f"worker{i}: port {port} を起動  log={log}  flags={flags}")

for i, srv in enumerate(SERVERS):
    INFO = None
    for _ in range(120):
        if PROCS[i].poll() is not None:
            raise RuntimeError(
                f"worker{i} のプロセスが終了した — !tail -50 {LOGS[i]} で確認。"
                f"ログの末尾が Killed ならホストRAM不足なので WORKERS を減らしてセル1→6をやり直す")
        try:
            INFO = json.load(urllib.request.urlopen(f"http://{srv}/object_info", timeout=5))
            break
        except Exception:
            time.sleep(2)
    assert INFO, f"worker{i} ({srv}) が起動しない — !tail -50 {LOGS[i]} で確認"
    h3_nodes = sorted(k for k in INFO if k.startswith("MiniMaxH3"))
    assert h3_nodes, f"worker{i}: H3ノードが登録されていない — ComfyUIのバージョンを確認"
    print(f"worker{i} ready: {srv}  MiniMaxH3 nodes: {len(h3_nodes)}")
print("nodes:", h3_nodes)

In [ ]:
#@title 7. チャプター生成（ユニット別にまとめてWORKERS本を同時実行→完了ごとに即退避。生成済みはスキップ＝中断・再開に強い）
import concurrent.futures, glob, json, os, queue, re, shutil, subprocess, sys, threading, time
if "wait_weights" in globals():
    wait_weights()  # セル3のバックグラウンド重み配置の完了を待つ
if not CHAPTERS:  # 未指定 = バンドル内の全チャプター
    CHAPTERS = [os.path.basename(w)[: -len("_workflow.json")] for w in glob.glob(f"{BUNDLE}/ch*_workflow.json")]
    print("CHAPTERS未指定 → 全チャプターを生成")
assert CHAPTERS, f"{BUNDLE} に ch*_workflow.json が無い — バンドルを確認"

AB = str(globals().get("AB_LABEL") or "")
def out_name(ch):
    # A/B計測時はラベル付き別名にして、生成済みスキップと衝突せずに同一チャプターを再生成できるようにする
    return f"{ch}__{AB}.mp4" if AB else f"{ch}.mp4"

# --- 計測ログ（A/B比較用。通常運用でも1行/チャプター追記され、セッションを跨いで蓄積する） ---
BENCH_CSV = "/content/outputs/bench_log.csv"
BENCH_LOCK = threading.Lock()

def last_step_sec(logpath):
    # ComfyUIログ末尾のtqdm行（例 "20/20 [27:00<00:00, 81.00s/it]"）からサンプリング1stepの秒数を拾う
    try:
        with open(logpath, "rb") as f:
            f.seek(max(0, os.path.getsize(logpath) - 65536))
            txt = f.read().decode(errors="ignore")
    except OSError:
        return ""
    hits = re.findall(r"([0-9.]+)\s*(s/it|it/s)", txt)
    if not hits:
        return ""
    v, unit = hits[-1]
    sec = float(v) if unit == "s/it" else (1.0 / float(v) if float(v) else 0.0)
    return f"{sec:.1f}"

def bench_append(ch, wall_sec, step_sec):
    with BENCH_LOCK:
        drive_csv = os.path.join(OUT_DRIVE_DIR, "bench_log.csv") if OUT_DRIVE_DIR else None
        if not os.path.exists(BENCH_CSV) and drive_csv and os.path.exists(drive_csv):
            shutil.copy(drive_csv, BENCH_CSV)  # 前セッションの計測に追記して、セッションを跨いで比較できるようにする
        new = not os.path.exists(BENCH_CSV)
        with open(BENCH_CSV, "a") as f:
            if new:
                f.write("label,chapter,gpu,workers,sage,fast_flags,comfy_flags,wall_sec,sampling_s_per_step\n")
            f.write(f"{AB or 'default'},{ch},{NAME.replace(',', ' ')},{WORKERS},"
                    f"{int(bool(globals().get('SAGE_ATTENTION')))},"
                    f"{' '.join(globals().get('FAST_FLAGS') or []) or '-'},"
                    f"{' '.join(COMFY_FLAGS) or '-'},{wall_sec:.0f},{step_sec or '-'}\n")
        if drive_csv:
            shutil.copy(BENCH_CSV, drive_csv)

def chapter_units(ch):
    with open(os.path.join(BUNDLE, f"{ch}_workflow.json")) as f:
        g = json.load(f)
    return sorted({v for node in g.values() for k, v in node.get("inputs", {}).items() if k == "unet_name"})

# 同じユニットを使うチャプターをグループにまとめる（21GBのユニット入れ替え回数を最小化。
# グループ単位で流すので、並列実行中にローカルへ置くユニットは常に1種類で済む）。
# 同一グループ内は番号順。生成順が変わるだけで成果物は同じ。
GROUPS = {}
for ch in sorted(CHAPTERS, key=lambda c: int(re.sub(r"\D", "", c) or 0)):
    GROUPS.setdefault(tuple(chapter_units(ch)), []).append(ch)
print(f"生成計画（{WORKERS}並列）:")
for units, chs in GROUPS.items():
    print("  ", " ".join(chs), "->", ", ".join(os.path.basename(u) for u in units))
os.makedirs("/content/outputs", exist_ok=True)
if OUT_DRIVE_DIR:
    os.makedirs(OUT_DRIVE_DIR, exist_ok=True)
# --- ディスク管理ヘルパ（逼迫時の自動掃除と、不足時に原因が分かる診断） ---
GiB = 2**30
COPY_FLOOR = 2 * GiB   # 大物コピー中に残しておきたい空き（DriveFSの読み出しキャッシュ用ヘッドルーム）
DIFF_DIR = "/content/ComfyUI/models/diffusion_models"
DRIVEFS_CACHE_DIRS = ["/root/.config/Google/DriveFS", "/root/.cache/Google/DriveFS"]

def headroom_for(remaining):
    # コピーに必要な空き = 残りバイト ＋ ヘッドルーム。ヘッドルームは残りに応じて縮める
    # （残り1GiBのレジューム時に「4GiB空いていないから不足」と落ちるのを防ぐ）。
    return remaining + min(COPY_FLOOR, max(remaining // 4, 512 * 2**20))

def free_bytes():
    return shutil.disk_usage("/content").free

def path_bytes(p):
    if os.path.isfile(p) and not os.path.islink(p):
        return os.path.getsize(p)
    total = 0
    for root, _d, fs in os.walk(p, onerror=lambda e: None):
        for f in fs:
            try:
                total += os.lstat(os.path.join(root, f)).st_size
            except OSError:
                pass
    return total

def deleted_open_bytes():
    # 削除済みなのにプロセスが開いたままのファイルは、dfの空きとして戻ってこない（ComfyUIが
    # unetをmmapしたまま等）。「消したのに空きが増えない」ときの原因を特定するために合計を見る。
    total, seen = 0, set()
    for fd in glob.glob("/proc/[0-9]*/fd/*"):
        try:
            if not os.readlink(fd).endswith(" (deleted)"):
                continue
            st = os.stat(fd)
        except OSError:
            continue
        if (st.st_dev, st.st_ino) in seen:
            continue
        seen.add((st.st_dev, st.st_ino))
        total += st.st_size
    return total

def biggest_files(n=6, roots=("/content", "/root"), floor=512 * 2**20):
    hits = []
    for r in roots:
        for root, dirs, fs in os.walk(r, onerror=lambda e: None):
            dirs[:] = [d for d in dirs if os.path.join(root, d) != "/content/drive"]  # Drive側は数えない
            for f in fs:
                p = os.path.join(root, f)
                try:
                    sz = os.lstat(p).st_size
                except OSError:
                    continue
                if sz >= floor:
                    hits.append((sz, p))
    return sorted(hits, reverse=True)[:n]

def disk_short_msg(need, remaining=None):
    # 「空きは残っているのにエラー」を避けるため、判断に使った数字をそのまま出す。
    du = shutil.disk_usage("/content")
    msg = (f"ディスク不足: 空き {du.free / GiB:.1f} GiB / 全体 {du.total / GiB:.1f} GiB、"
           f"必要 {need / GiB:.1f} GiB")
    if remaining is not None:
        msg += f"（コピー残り {remaining / GiB:.1f} GiB ＋ 作業用ヘッドルーム）"
    dob = deleted_open_bytes()
    if dob >= GiB:
        msg += (f"\n  → 削除済みなのにプロセスが掴んでいるファイルが {dob / GiB:.1f} GiB ある"
                "（ComfyUIがunetをmmapしたまま等）。セル6を再実行してComfyUIを再起動すれば解放される")
    big = biggest_files()
    if big:
        msg += "\n  → ローカルの大きいファイル: " + ", ".join(f"{p}={s / GiB:.1f}GiB" for s, p in big)
    return msg + "\n  → 不要ファイルを整理してこのセルを再実行（コピーは途中から再開する）"

def purge_drivefs_cache():
    # DriveFSはFUSE読み出しの内容キャッシュをローカルディスクにも書くため、大物コピー中に
    # 「コピー先＋キャッシュ」の二重消費でディスクが枯渇することがある（ENOSPC実測・2026-08）。
    # unmount→キャッシュ削除→remountで空ける。掃除で実際に何GiB空いたかを必ず表示する
    # （0GiBなら原因はキャッシュではない＝重み本体か、プロセスが掴んだ削除済みファイル）。
    from google.colab import drive as _gd
    lock = globals().get("DRIVE_IO_LOCK")
    if lock:
        lock.acquire()
    try:
        before = free_bytes()
        hit = [d for d in DRIVEFS_CACHE_DIRS if os.path.isdir(d)]
        print(f"  空きディスク逼迫（空き {before / GiB:.1f} GiB）→ DriveFSキャッシュを掃除"
              f"（unmount→削除→remount・数十秒）: {', '.join(hit) or '(キャッシュ無し)'}", flush=True)
        _gd.flush_and_unmount()
        for d in hit:
            shutil.rmtree(d, ignore_errors=True)
        _gd.mount("/content/drive")
        after = free_bytes()
        print(f"  キャッシュ掃除で {max(0, after - before) / GiB:.1f} GiB 解放"
              f"（空き {after / GiB:.1f} GiB）", flush=True)
        return after
    finally:
        if lock:
            lock.release()

def reclaim_disk(need, keep=()):
    # needバイトの空きを作る。安いものから順に捨て、都度measureし直して足りたら止める
    # （1手が効かなかったときに次の手へ進めるようにする）。戻り値は最終的な空きバイト。
    free = free_bytes()
    if free >= need:
        return free
    # 1) 他のunetのローカル実体（Driveに実体があるので消してよい。symlinkは実体を持たないので対象外）
    for other in sorted(glob.glob(f"{DIFF_DIR}/*.safetensors") + glob.glob(f"{DIFF_DIR}/*.part")):
        if free >= need:
            return free
        if other in keep or os.path.islink(other):
            continue
        sz = path_bytes(other)
        os.remove(other)
        print(f"  ディスク確保のためローカルunetを削除（Driveに実体あり）: "
              f"{os.path.basename(other)} -{sz / GiB:.1f} GiB", flush=True)
        free = free_bytes()
    # 2) 生成に不要なキャッシュ・作業ファイル
    junk_list = ["/root/.cache/pip", "/root/.cache/huggingface", "/root/.cache/torch",
                 "/content/h3_dl", "/content/ComfyUI/temp"]
    if glob.glob("/content/bundle/**/script.md", recursive=True):  # 展開済みならバンドルzipは不要
        junk_list += [z for z in sorted(glob.glob("/content/*.zip"))
                      if os.path.basename(z) != "h3_outputs.zip"]  # 回収用zipは消さない
    for junk in junk_list:
        if free >= need:
            return free
        if junk in keep or not os.path.exists(junk):
            continue
        sz = path_bytes(junk)
        if sz < 200 * 2**20:
            continue
        if os.path.isdir(junk) and not os.path.islink(junk):
            shutil.rmtree(junk, ignore_errors=True)
        else:
            os.remove(junk)
        print(f"  ディスク確保のため削除: {junk} -{sz / GiB:.1f} GiB", flush=True)
        free = free_bytes()
    # 3) DriveFSの読み出しキャッシュ（remountに数十秒かかるので最後）
    if free < need:
        free = purge_drivefs_cache()
    return free

def copy_from_drive(src, dst, threads=8, chunk=64 * 2**20):
    # Drive→ローカルのレジューム可能コピー。並列pread（8スレッド・実測62→83MB/s）で読み、
    # 追記順を守って書く＝dstのファイルサイズがそのままレジューム点になる。
    # 空きが逼迫したらキャッシュ掃除等で空けて続行する（shutil.copyだとENOSPCで落ちる）。
    size = os.path.getsize(src)
    done = os.path.getsize(dst) if os.path.exists(dst) else 0

    def read_round(start):  # startからthreads*chunk分を並列preadで読んで返す
        n = min(threads * chunk, size - start)
        def one(i):
            off, ln = start + i * chunk, min(chunk, n - i * chunk)
            if ln <= 0:
                return b""
            fd = os.open(src, os.O_RDONLY)
            try:
                parts, got = [], 0
                while got < ln:
                    b = os.pread(fd, ln - got, off + got)
                    assert b, f"{src} の読み出しが途切れた — セルの再実行で続きから再開する"
                    parts.append(b)
                    got += len(b)
                return b"".join(parts)
            finally:
                os.close(fd)
        with concurrent.futures.ThreadPoolExecutor(threads) as ex:
            return b"".join(ex.map(one, range(threads)))

    with concurrent.futures.ThreadPoolExecutor(1) as ahead:
        nxt = None  # 先読み: 書き込みと次ラウンドの読みを重ねる
        while done < size:
            buf = nxt.result() if nxt else None
            nxt = None
            # 必要な空きは「残りのバイト＋ヘッドルーム」。残りが少ないときにヘッドルームを
            # 理由に止めない（空きが残っているのに落ちるのを防ぐ）。
            want = headroom_for(size - done)
            if free_bytes() < want:
                free = reclaim_disk(want, keep=(dst,))
                assert free >= want, disk_short_msg(want, size - done)
            if buf is None:
                buf = read_round(done)
            if done + len(buf) < size:
                nxt = ahead.submit(read_round, done + len(buf))
            with open(dst, "ab") as fo:
                fo.write(buf)
            if done // (4 * 2**30) != (done + len(buf)) // (4 * 2**30):
                print(f"    ... {(done + len(buf)) / 2**30:.0f}/{size / 2**30:.0f} GiB", flush=True)
            done += len(buf)

def ensure_local_unet(ch):
    # 実行時の重みはローカル実体が必須（Drive FUSE越しのsymlinkはcomfy_aimdoが読めない）。
    # このチャプターが使うunetをGCS（あれば優先・数倍速い）またはDriveからローカル化し、
    # ディスクが足りなければ他のunetのローカル実体・不要キャッシュ・DriveFSキャッシュを
    # 順に捨てて空ける（reclaim_disk）。実体は全ワーカーで共有されるので、グループを流す前に1回だけ呼べばよい。
    for u in chapter_units(ch):
        pth = f"{DIFF_DIR}/{u}"
        drv = f"{WEIGHTS_DRIVE_DIR}/{u}" if WEIGHTS_DRIVE_DIR else None
        expected = ((globals().get("GCS_SIZES") or {}).get(u)
                    or (os.path.getsize(drv) if drv and os.path.exists(drv) else None))
        if os.path.exists(pth) and not os.path.islink(pth):
            if expected is None or os.path.getsize(pth) == expected:
                continue  # 完全なローカル実体
            os.rename(pth, pth + ".part")  # 書きかけ（前回のENOSPC等）→ レジュームで続きから
        part = pth + ".part"
        if globals().get("gcs_fetch") and gcs_fetch(u, pth):
            if os.path.exists(part):
                os.remove(part)  # GCSから取り直したのでDrive用の書きかけは不要
            print("  local OK（GCS）", u, flush=True)
            continue
        assert drv and os.path.exists(drv), f"{u} が未配置（GCS/Driveのどちらにも無い）— セル3を実行"
        remaining = os.path.getsize(drv) - (os.path.getsize(part) if os.path.exists(part) else 0)
        need = headroom_for(remaining)
        free = reclaim_disk(need, keep=(pth, part))
        assert free >= need, disk_short_msg(need, remaining)
        print(f"  {u} をDriveからローカル化中（残り {remaining / GiB:.1f} GiB・空き {free / GiB:.1f} GiB・"
              "8スレッド並列コピー。中断してもレジューム可）...", flush=True)
        copy_from_drive(drv, part)
        if os.path.lexists(pth):
            os.remove(pth)
        os.replace(part, pth)
        print("  local OK", u)

PRINT_LOCK = threading.Lock()   # 2本分の進捗が行の途中で混ざらないようにする
DRIVE_OUT_LOCK = threading.Lock()

def run_chapter(ch, server, tag, wlog):
    wf = os.path.join(BUNDLE, f"{ch}_workflow.json")
    assert os.path.exists(wf), f"{wf} が無い"
    out = f"/content/outputs/{out_name(ch)}"
    t0 = time.time()
    with PRINT_LOCK:
        print(f"[{tag}] === {ch} 生成開始 @ {server}（ポーリング出力が続いていれば正常）===", flush=True)
    # -u で子プロセスの出力をバッファさせず、行が届き次第ワーカー名を付けて流す
    p = subprocess.Popen(
        [sys.executable, "-u", os.path.join(BUNDLE, "h3_run.py"), wf, "--out", out, "--server", server],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        with PRINT_LOCK:
            print(f"[{tag}] {ch}: {line.rstrip()}", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"{ch} の生成が失敗")
    wall = time.time() - t0
    step_sec = last_step_sec(wlog)
    bench_append(ch, wall, step_sec)
    with PRINT_LOCK:
        print(f"[{tag}] {ch}: ⏱ {wall / 60:.1f}分（サンプリング1step≒{step_sec or '?'}秒）→ bench_log.csv に記録", flush=True)
    if OUT_DRIVE_DIR:
        with DRIVE_OUT_LOCK:
            shutil.copy(out, OUT_DRIVE_DIR)
        with PRINT_LOCK:
            print(f"[{tag}] {ch}: -> Drive退避済み: {OUT_DRIVE_DIR}/{out_name(ch)}", flush=True)

FAILED = []
for units, chs in GROUPS.items():
    todo = []
    for ch in chs:
        if os.path.exists(f"/content/outputs/{out_name(ch)}"):
            print("skip（生成済み）", ch)
        else:
            todo.append(ch)
    if not todo:
        continue
    ensure_local_unet(todo[0])  # グループ内は同じunet — ローカル化は1回でよい
    pending = queue.Queue()
    for ch in todo:
        pending.put(ch)

    def worker(i):
        while True:
            try:
                ch = pending.get_nowait()
            except queue.Empty:
                return
            try:
                run_chapter(ch, SERVERS[i], f"w{i}", LOGS[i])
            except Exception as e:  # 1本コケても残りは流し切る（成功分は回収できる）
                FAILED.append(ch)
                with PRINT_LOCK:
                    print(f"[w{i}] ⚠ {ch} 失敗: {e} — !tail -80 {LOGS[i]} で確認", flush=True)

    ts = [threading.Thread(target=worker, args=(i,)) for i in range(min(WORKERS, len(todo)))]
    for t in ts:
        t.start()
    for t in ts:
        t.join()

assert not FAILED, (f"失敗したチャプター: {FAILED} — 各ワーカーのログ（{', '.join(LOGS)}）を確認。"
                    " OOMなら WORKERS を減らすか COMFY_FLAGS=['--lowvram']。"
                    " このセルを再実行すれば成功済みはスキップされる")
print("指定チャプター完了。ブラウザにも落とすならセル8へ")
if os.path.exists(BENCH_CSV):
    print("⏱ 計測ログ（bench_log.csv・A/B比較はここを見る）:")
    print(open(BENCH_CSV).read().rstrip())

# --- 自動切断（AUTO_SHUTDOWN=True のときだけ） ---
# ここに到達している＝指定チャプターは全て成功。ただしランタイムを削除すると /content は丸ごと
# 消えるので、「指定した全チャプターのmp4がDrive上に実在する」ことを確認できたときだけ切断する。
if globals().get("AUTO_SHUTDOWN"):
    not_on_drive = [ch for ch in CHAPTERS
                    if not os.path.exists(os.path.join(OUT_DRIVE_DIR, out_name(ch)))]
    if not_on_drive:
        print(f"⚠ 自動切断を中止: Driveに見つからないチャプターがある {not_on_drive} —"
              " セル8で回収してから手動でランタイムを削除すること", flush=True)
    else:
        sizes = {ch: os.path.getsize(os.path.join(OUT_DRIVE_DIR, out_name(ch))) for ch in CHAPTERS}
        print(f"★ 全{len(CHAPTERS)}チャプターのDrive退避を確認: "
              + ", ".join(f"{ch}={s / 2**20:.1f}MB" for ch, s in sizes.items()), flush=True)
        print(f"★ {SHUTDOWN_GRACE_SEC}秒後にランタイムを切断・削除して課金を止める。"
              "取り消すならこのセルを停止（■）", flush=True)
        for left in range(SHUTDOWN_GRACE_SEC, 0, -15):
            print(f"   切断まで {left}秒...", flush=True)
            time.sleep(min(15, left))
        try:
            from google.colab import drive as _gd
            _gd.flush_and_unmount()  # Driveへの書き込みを確実に反映させてから消す
            print("   Driveへ書き切った", flush=True)
        except Exception as e:
            print(f"   ⚠ flush_and_unmount に失敗（mp4のDrive実在は確認済み）: {e}", flush=True)
        print("   ランタイムを削除する。以降の出力は表示されない", flush=True)
        from google.colab import runtime
        runtime.unassign()

In [ ]:
#@title 8. 成果物の回収（zip→ブラウザDL。Drive退避済みならスキップ可）
import glob, subprocess
outs = sorted(glob.glob("/content/outputs/*.mp4"))
assert outs, "/content/outputs にmp4が無い"
print(*outs, sep="\n")
subprocess.run(["zip", "-j", "-q", "/content/h3_outputs.zip", *outs], check=True)
from google.colab import files
files.download("/content/h3_outputs.zip")
print("回収したら「ランタイム → ランタイムを接続解除して削除」で課金を止めること")


In [ ]:
#@title 9.（任意）アドホック生成 — 画像・音声・プロンプトを直接指定して1本作る
# チャプター定義に縛られない単発生成。素材はバンドル同梱ファイル名で指定（新素材は左のファイルペインで
# /content/ComfyUI/input/ へドラッグ＆ドロップしてから指定）。framesは17k+5グリッド（90,124,141,158,...）。
# H3の埋め込み音声は入力wavと同等（2026-08実測）なので、出力の音声はそのまま最終成果物に使える。
# 単発なので常にworker0（SERVERS[0]）で走らせる。
ADHOC = dict(
    mode="r2v",      # "i2v"=開始/終了フレーム固定・音声なし / "r2v"=参照画像(≦9)+音声(≦3・各2〜15s)・リップシンク
    frames=124,
    prompt="Required attached input files: <Picture 1> = XXX.png — ...; <Audio 1> = YYY.wav — spoken line, use AS-IS. "
           "The video starts EXACTLY on <Picture 1>. ... (S1) speaks — he says <d>[Japanese] セリフ</d>, lip-syncing to <Audio 1>. "
           "Soundscape: ... Music: no background music.",
    first="chN_start.png", last="chN_end.png",  # i2vのみ
    images=["XXX.png"], audio=["YYY.wav"],      # r2vのみ（<Picture N>/<Audio N>の接続順）
    out="adhoc1",
)
import json, os, subprocess, sys
pf = os.path.join(BUNDLE, f"{ADHOC['out']}_prompt.txt")
with open(pf, "w") as f:
    f.write(ADHOC["prompt"])
wf = os.path.join(BUNDLE, f"{ADHOC['out']}_workflow.json")
cmd = [sys.executable, os.path.join(BUNDLE, "build_h3_workflow.py"), "--mode", ADHOC["mode"],
       "--out", wf, "--prompt-file", pf, "--frames", str(ADHOC["frames"]),
       "--prefix", f"video/{ADHOC['out']}",
       "--encoder", ENCODER, "--unet-i2v", UNET_I2V, "--unet-r2v", UNET_R2V]
if ADHOC["mode"] == "i2v":
    cmd += ["--first", ADHOC["first"], "--last", ADHOC["last"]]
else:
    for im in ADHOC["images"]:
        cmd += ["--image", im]
    for au in ADHOC["audio"]:
        cmd += ["--audio", au]
subprocess.run(cmd, check=True)
with open(wf) as f:  # 旧builder対策のcodec補完
    _d = json.load(f)
for _n in _d.values():
    if _n.get("class_type") == "SaveVideo":
        _n["inputs"].setdefault("codec", "auto")
        _n["inputs"].setdefault("format", "auto")
with open(wf, "w") as f:
    json.dump(_d, f, indent=1)
if "wait_weights" in globals():
    wait_weights()  # セル3のバックグラウンド重み配置の完了を待つ
if "ensure_local_unet" in globals():
    ensure_local_unet(ADHOC["out"])  # 使用unetのローカル実体を確保（セル7と同じ仕組み）
os.makedirs("/content/outputs", exist_ok=True)
r = subprocess.run([sys.executable, os.path.join(BUNDLE, "h3_run.py"), wf,
                    "--out", f"/content/outputs/{ADHOC['out']}.mp4", "--server", SERVERS[0]])
print("結果:", "完了 -> セル8で回収" if r.returncode == 0 else f"失敗 — !tail -80 {LOGS[0]}")

In [ ]:
#@title 10.（任意）高速化フラグの全パターン計測 — このセル1つでセットアップ→全パターン生成→比較表まで
#@markdown パイロットチャプター1本を、GPUに応じた全フラグパターン（**A100=2通り** base/sage、**L4=4通り** base/fast/sage/sage_fast）で
#@markdown **同一シード**生成して所要時間を計測する。**最後に出る比較表をそのままClaudeへ貼れば、採用フラグを相談できる。**
#@markdown - 事前にセル1の設定（Driveパス・`EXPECTED_GPU`・`WORKERS`等）だけ済ませておく。セットアップ（セル1→3→4→2→5）はこのセルが内部で実行する
#@markdown - 生成済みパターンはスキップされ、以前の計測値（bench_log.csv）が表に使われる＝中断後の再実行で続きから
#@markdown - `BENCH_CHAPTER`: 計測に使うチャプター（例 "ch1"）。空 = バンドル先頭のチャプター
#@markdown - `BENCH_SHUTDOWN`: 全パターン完了後、成果物とbench_log.csvのDrive退避を確認してからランタイムを削除する（離席用。比較表はセル出力に残る）
BENCH_CHAPTER = ""  #@param {type:"string"}
BENCH_SHUTDOWN = False  #@param {type:"boolean"}

BENCH_CELL_MARKER = True  # このセル自身をセル番号実行の対象から外す目印（消さない）
import csv as _bm_csv, glob as _bm_glob, os as _bm_os, re as _bm_re, time as _bm_time

def _bm_load_steps():
    # ★一括実行セルと同じ方法で、ノートブック自身のセルソースを「#@title <番号>.」で取り出す
    from google.colab import _message
    res = _message.blocking_request("get_ipynb", timeout_sec=120)
    nb = res["ipynb"] if isinstance(res, dict) and "ipynb" in res else res
    steps = {}
    for c in nb["cells"]:
        if c.get("cell_type") != "code":
            continue
        src = c["source"]
        src = src if isinstance(src, str) else "".join(src)
        if "BENCH_CELL_MARKER" in src:
            continue
        for line in src.split("\n")[:3]:
            m = _bm_re.match(r"\s*(?:#@title|#)\s*(\d)[.．]", line)
            if m:
                steps.setdefault(int(m.group(1)), src)
                break
    return steps

_bm_ip = get_ipython()
try:
    _BM_STEPS = _bm_load_steps()
except Exception as _bm_e:
    raise SystemExit(f"この環境ではノートブックのセルを取り出せなかった（{type(_bm_e).__name__}: {_bm_e}）"
                     " — セル1〜6を手で実行し、AB_LABELを変えながらセル6→7を繰り返す手動A/Bにフォールバックする")

def _bm_run(n):
    res = _bm_ip.run_cell(_BM_STEPS[n])
    if not res.success:
        err = getattr(res, "error_in_exec", None) or getattr(res, "error_before_exec", None)
        raise RuntimeError(f"セル{n} で失敗: {err!r}" if err is not None
                           else f"セル{n} で失敗（原因は直前のトレースバック）")

def _bm_recover_drive():
    # Drive FUSEの瞬断（[Errno 107] Transport endpoint is not connected 等・2026-08実測）からの復旧。
    # 壊れたマウントは /proc/mounts に残ったまま「マウント済み」に見えることがあるので、強制再マウントで張り直す
    from google.colab import drive as _gd
    try:
        _gd.flush_and_unmount()
    except Exception:
        pass  # マウントが壊れているとここ自体が失敗するが、force_remount で張り直せる
    _gd.mount("/content/drive", force_remount=True)

_BM_WEIGHT_ERRS = ("重み配置", "Transport endpoint", "Errno 107", "未配置")

def _bm_ensure_weights():
    # セル3のバックグラウンド重み配置の完了確認。Drive瞬断等で失敗していたら、
    # 再マウント→セル3再実行（コピー/DLは途中から再開）で最大2回まで自動復旧する
    for _bm_at in (1, 2):
        try:
            G["wait_weights"]()
            return
        except BaseException as _bm_we:
            print(f"⚠ 重み配置の失敗を検出（{_bm_we}）— Driveを再マウントしてセル3を再実行する"
                  f"（自動復旧 {_bm_at}/2回目・コピーは途中から再開）", flush=True)
            _bm_recover_drive()
            _bm_run(3)
    G["wait_weights"]()  # 2回復旧しても駄目なら例外のまま停止

def _bm_premount():
    # Driveのマウント承認（操作が必要）を先に済ませる — ★一括実行セルと同じ
    if not any(globals().get(k) for k in ("WEIGHTS_DRIVE_DIR", "BUNDLE_ZIP_FROM_DRIVE", "OUT_DRIVE_DIR")):
        return
    if _bm_os.path.isdir("/content/drive/MyDrive"):
        return
    from google.colab import drive as _gd
    print("★ 先にGoogle Driveをマウントする（初回は承認ダイアログが出る）", flush=True)
    _gd.mount("/content/drive")

def _bm_preupload():
    # バンドルzipのアップロード（操作が必要）を先に済ませる — ★一括実行セルと同じ
    if globals().get("BUNDLE_ZIP_FROM_DRIVE"):
        return
    cur = globals().get("BUNDLE_ZIP_LOCAL") or ""
    if cur and _bm_os.path.exists(cur):
        return
    from google.colab import files
    print("★ 先にバンドルzip（<NN>_<slug>_bundle.zip）をアップロードする — 操作が必要なのはここだけ", flush=True)
    up = files.upload()
    zips = [n for n in up if n.lower().endswith(".zip")]
    assert zips, "zipが選ばれていない — バンドルzipを選ぶか、セル1で BUNDLE_ZIP_FROM_DRIVE を設定する"
    globals()["BUNDLE_ZIP_LOCAL"] = _bm_os.path.join(_bm_os.getcwd(), zips[0])

G = globals()
_bm_t0 = _bm_time.time()
print("★ 全パターン計測: セットアップ（セル1→3→4→2→5）から始める", flush=True)
_bm_run(1)
assert G.get("VRAM", 0) > 0, "GPUランタイムでない — 計測できない（ランタイム → ランタイムのタイプを変更 → L4/A100）"
G["AUTO_SHUTDOWN"] = False  # パターン毎にセル7が切断しないよう強制する（切断は最後に BENCH_SHUTDOWN で行う）
_bm_premount()
_bm_preupload()
for _bm_n in (3, 4, 2, 5):  # ★一括実行と同じ順（セル3の重みコピーをpip install・バンドル展開と並走させる）
    _bm_run(_bm_n)
_bm_ensure_weights()  # パターン開始前に重み配置を確定させる（Drive瞬断で失敗していれば自動復旧）

_BM_FAST = ["fp16_accumulation", "fp8_matrix_mult"]
if G["CAP"] >= (8, 9):  # Ada/Hopper: --fast系が効くので4パターン
    _BM_PATTERNS = [("base", False, []), ("fast", False, _BM_FAST),
                    ("sage", True, []), ("sage_fast", True, _BM_FAST)]
else:                   # Ampere(A100): --fast系は効かないので2パターン
    _BM_PATTERNS = [("base", False, []), ("sage", True, [])]

if not BENCH_CHAPTER:
    _bm_wfs = sorted(_bm_glob.glob(f"{G['BUNDLE']}/ch*_workflow.json"),
                     key=lambda p: int(_bm_re.sub(r"\D", "", _bm_os.path.basename(p)) or 0))
    assert _bm_wfs, f"{G['BUNDLE']} に ch*_workflow.json が無い — バンドルを確認"
    BENCH_CHAPTER = _bm_os.path.basename(_bm_wfs[0])[: -len("_workflow.json")]
assert _bm_os.path.exists(f"{G['BUNDLE']}/{BENCH_CHAPTER}_workflow.json"), f"{BENCH_CHAPTER} のworkflowが無い"
print(f"\n★ 計測パターン: {[p[0] for p in _BM_PATTERNS]}  チャプター: {BENCH_CHAPTER}", flush=True)
print("  1パターン = チャプター1本の生成（L4実測: 6秒級で40分弱/本 → 4パターンで2.5時間前後。"
      "A100は2パターンで従来比の実測がそのまま取れる）", flush=True)

_bm_results = {}
for _bm_label, _bm_sage, _bm_fastf in _BM_PATTERNS:
    print(f"\n{'=' * 78}\n▶ パターン {_bm_label}（SAGE_ATTENTION={_bm_sage} FAST_FLAGS={_bm_fastf or '[]'}）"
          f"（経過 {(_bm_time.time() - _bm_t0) / 60:.1f}分）\n{'=' * 78}", flush=True)
    G["SAGE_ATTENTION"], G["FAST_FLAGS"], G["AB_LABEL"] = _bm_sage, list(_bm_fastf), _bm_label
    G["CHAPTERS"] = [BENCH_CHAPTER]
    G["AUTO_SHUTDOWN"] = False
    try:
        _bm_run(6)  # フラグはComfyUIの起動引数なので、パターン毎に再起動して反映する
        try:
            _bm_run(7)
        except BaseException as _bm_e:
            # 生成中のDrive瞬断・重み未配置は自動復旧して1回だけやり直す（生成済みチャプターは
            # セル7がスキップするので、やり直しは失敗地点からの再開になる）
            if not any(k in str(_bm_e) for k in _BM_WEIGHT_ERRS):
                raise
            print(f"⚠ Drive/重み起因の失敗（{_bm_e}）— 自動復旧して {_bm_label} をやり直す", flush=True)
            _bm_recover_drive()
            _bm_ensure_weights()
            _bm_run(7)
        _bm_results[_bm_label] = "OK"
    except KeyboardInterrupt:
        raise
    except BaseException as _bm_e:
        _bm_results[_bm_label] = f"失敗: {_bm_e}"
        for _bm_lg in sorted(_bm_glob.glob("/content/comfyui_*.log")):
            try:
                with open(_bm_lg, errors="replace") as _bm_fo:
                    print(f"--- {_bm_lg}（末尾） ---\n{_bm_fo.read()[-2000:]}", flush=True)
            except OSError:
                pass
        print(f"⚠ パターン {_bm_label} が失敗 — 残りのパターンは続行する", flush=True)

# --- 比較表（bench_log.csv の各ラベル最新行から作る） ---
_bm_best = {}
if _bm_os.path.exists("/content/outputs/bench_log.csv"):
    for _bm_r in _bm_csv.DictReader(open("/content/outputs/bench_log.csv")):
        if _bm_r["chapter"] == BENCH_CHAPTER:
            _bm_best[_bm_r["label"]] = _bm_r  # 後の行ほど新しい = 最後の計測を採用
_bm_base = _bm_best.get("base")
print(f"\n{'=' * 78}", flush=True)
print(f"★ 計測結果 — GPU {G['NAME']} / WORKERS {G['WORKERS']} / チャプター {BENCH_CHAPTER}"
      f" / 合計 {(_bm_time.time() - _bm_t0) / 60:.1f}分", flush=True)
print("★ 下の表をそのままClaudeへ貼り付けて、採用フラグを相談する", flush=True)
print(f"\n{'pattern':<11}{'wall_min':>9}{'s_per_step':>11}{'vs_base':>9}  status", flush=True)
for _bm_label, _sg, _ff in _BM_PATTERNS:
    _bm_r = _bm_best.get(_bm_label)
    _bm_st = _bm_results.get(_bm_label, "")
    if not _bm_r:
        print(f"{_bm_label:<11}{'-':>9}{'-':>11}{'-':>9}  {_bm_st or '記録なし'}", flush=True)
        continue
    _bm_wall = float(_bm_r["wall_sec"])
    if _bm_label == "base":
        _bm_ratio = "x1.00"
    elif _bm_base and _bm_wall:
        _bm_ratio = f"x{float(_bm_base['wall_sec']) / _bm_wall:.2f}"
    else:
        _bm_ratio = "-"
    print(f"{_bm_label:<11}{_bm_wall / 60:>9.1f}{_bm_r['sampling_s_per_step']:>11}{_bm_ratio:>9}  {_bm_st}",
          flush=True)
print(f"\n品質比較: /content/outputs/{BENCH_CHAPTER}__<パターン>.mp4 を見比べる（同一シードなので条件差だけが出る）"
      + (f"。Drive退避先: {G['OUT_DRIVE_DIR']}" if G.get("OUT_DRIVE_DIR") else ""), flush=True)
print("本番ランへ戻るときはセル1から実行し直す（CHAPTERS・AUTO_SHUTDOWN等をこのセルが書き換えている）", flush=True)

# --- 離席用の自動切断（成果物と計測ログのDrive退避を確認できたときだけ） ---
if BENCH_SHUTDOWN:
    _bm_out = G.get("OUT_DRIVE_DIR") or ""
    _bm_unsaved = [_bm_os.path.basename(o) for o in sorted(_bm_glob.glob("/content/outputs/*.mp4"))
                   if not (_bm_out and _bm_os.path.exists(_bm_os.path.join(_bm_out, _bm_os.path.basename(o))))]
    if not _bm_out or _bm_unsaved or not _bm_os.path.exists(_bm_os.path.join(_bm_out, "bench_log.csv")):
        print(f"⚠ 自動切断を中止: Drive退避を確認できない（OUT_DRIVE_DIR={_bm_out or '未設定'}"
              f" 未退避={_bm_unsaved}）— セル8で回収してから手動でランタイムを削除すること", flush=True)
    else:
        print("★ 60秒後にランタイムを切断・削除して課金を止める（比較表はこのセルの出力に残る）。"
              "取り消すならこのセルを停止（■）", flush=True)
        _bm_time.sleep(60)
        try:
            from google.colab import drive as _gd
            _gd.flush_and_unmount()  # Driveへの書き込み（mp4・bench_log.csv）を確実に反映させてから消す
        except Exception as _bm_e:
            print(f"⚠ flush_and_unmount に失敗: {_bm_e}", flush=True)
        from google.colab import runtime
        runtime.unassign()

## 後工程（ローカル）

回収した`chN.mp4`をラン専用ディレクトリに置き、ffmpegでconcat結合する（埋め込み音声をそのまま使う）。
エンドカード等の文字入れはCapCutで後付け（H3には文字を描かせない）。手順の詳細はランの`H3_COLAB.md`／
`.claude/skills/colab-video/SKILL.md`を参照。

トラブル時はセル出力と各ワーカーのログ（`!tail -80 /content/comfyui_0.log`、2並列なら
`/content/comfyui_1.log`も）をClaude Code / Cursorに貼れば診断できる。
